In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score
import xgboost as xgb
import lightgbm as lgb

df = pd.read_csv("netflix_engineered.csv")

In [6]:
# Predicting release_year, no leage concern here, so the full feature set is fair game.
feature_num = ['Year_Added', 'Month_Added', 'Cast_Count', 'Genre_Count', 'Duration_Value',
               'Title_Length', 'Description_WordCount', 'Has_Director', 'Has_Country']
feature_cat = ['type', 'rating', 'Primary_Country', 'Primary_Genre']

X = df[feature_num + feature_cat]
y = df['release_year']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), feature_num),
    ('cat', OneHotEncoder(handle_unknown='ignore'), feature_cat)
])


In [7]:
models = {
    "XGBoost Regressor": xgb.XGBRegressor(n_estimators=200, random_state=42),
    "LightGBM Regressor": lgb.LGBMRegressor(n_estimators=200, random_state=42, verbose=-1),
    "Neural Network (MLP) Regressor": MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42),
}

In [8]:
results = {}
for name, model in models.items():
    pipe = Pipeline([('pre', preprocessor),('reg', model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    mse = mean_squared_error(y_test,pred)
    results[name] ={"MSE": mse, "RMSE": mse ** 0.5, "R2": r2_score(y_test,pred)}
    print(f"{name:32s} MSE = {mse:.3f} RMSE = {mse ** 0.5:.3f} R@ = {r2_score(y_test,pred):.3f}")

XGBoost Regressor                MSE = 62.239 RMSE = 7.889 R@ = 0.335


C:\Users\sholy\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


LightGBM Regressor               MSE = 55.556 RMSE = 7.454 R@ = 0.407
Neural Network (MLP) Regressor   MSE = 81.532 RMSE = 9.029 R@ = 0.129


In [11]:
results_df = pd.DataFrame(results).T
result_df.to_csv("regression_results.csv")
print(results_df)

                                      MSE      RMSE        R2
XGBoost Regressor               62.239487  7.889201  0.335414
LightGBM Regressor              55.555733  7.453572  0.406782
Neural Network (MLP) Regressor  81.531785  9.029495  0.129413
